# Cluster Forecasting Experiment Runner

This notebook keeps the forecasting features built from raw 2023 daily usage data.
Cluster labels are only used to route households into cluster-specific models.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"
BRANCH = os.environ.get("NOTEBOOK_GIT_BRANCH", "extended-xgb-experiments")

kaggle_root = Path("/kaggle/working")
cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME and (cwd / "src" / "forecasting").exists():
    REPO_ROOT = cwd
elif kaggle_root.exists():
    repo_dir = kaggle_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    raise RuntimeError(
        "Could not determine REPO_ROOT automatically. Open the notebook from the repo root or run it on Kaggle."
    )

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)

Cloning into '/kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround'...


REPO_ROOT: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround
SRC_PATH: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround/src


Updating files: 100% (64/64), done.


In [2]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

from forecasting.data import (
    discover_cluster_cases,
    ensure_experiment_dirs,
    ensure_output_dirs,
    load_wide_csv,
)

OUT = ensure_output_dirs(REPO_ROOT)

TRAIN_23_PATH = REPO_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH = REPO_ROOT / "data" / "raw" / "sample_24.csv"
CLUSTER_CASE_DIR = REPO_ROOT / "notebooks" / "outputs" / "feature"
SHAPELET_STATIC_PATH = REPO_ROOT / "notebooks" / "outputs" / "shapelet_experiments" / "shapelet_features.csv"

In [3]:
DEBUG = False
DEBUG_FRAC = 0.2

MODEL_NAME = "xgb"
GPU_ENABLED = True
MIN_CLUSTER_ROWS = 500
MIN_CLUSTER_HOUSEHOLDS = 30

SELECTED_CASES = ["case5"]
INCLUDE_BASE_VARIANT = True
INCLUDE_SHAPELET_STATIC_VARIANT = False

XGB_PROFILE = "regularized"
XGB_PROFILES = ["regularized", "shallow", "deeper"]
XGB_PARAMS = None
RECURSIVE_VALIDATION_ENABLED = True
RECURSIVE_VALIDATION_DAYS = 60
CLUSTER_GATING_ENABLED = True
CLUSTER_GATE_MARGIN = 0.0
INCLUDE_PROFILE_FEATURES = True
INCLUDE_SEASONAL_PRIORS = True

PLOT_SAMPLE_PER_GROUP = 2
PLOT_MAX_GROUPS = None
RANDOM_STATE = 42

In [4]:
def build_experiment_configs(case_paths: dict[str, Path]) -> list[dict]:
    case_names = SELECTED_CASES or list(case_paths)
    missing = sorted(set(case_names) - set(case_paths))
    if missing:
        raise ValueError(f"Unknown case names requested: {missing}")

    variant_defs = []
    if INCLUDE_BASE_VARIANT:
        variant_defs.append(("base", None))
    if INCLUDE_SHAPELET_STATIC_VARIANT:
        if not SHAPELET_STATIC_PATH.exists():
            raise FileNotFoundError(f"Missing shapelet static feature file: {SHAPELET_STATIC_PATH}")
        variant_defs.append(("shapelet_static", SHAPELET_STATIC_PATH))

    configs = []
    for case_name in case_names:
        for variant_name, static_path in variant_defs:
            configs.append(
                {
                    "case_name": case_name,
                    "variant_name": variant_name,
                    "experiment_name": f"{case_name}_{variant_name}",
                    "cluster_path": str(case_paths[case_name]),
                    "static_features_path": str(static_path) if static_path else None,
                }
            )
    return configs


def build_run_settings() -> dict:
    return {
        "debug": DEBUG,
        "debug_frac": DEBUG_FRAC,
        "model_name": MODEL_NAME,
        "gpu_enabled": GPU_ENABLED,
        "min_cluster_rows": MIN_CLUSTER_ROWS,
        "min_cluster_households": MIN_CLUSTER_HOUSEHOLDS,
        "xgb_profile": XGB_PROFILE,
        "xgb_profiles": XGB_PROFILES,
        "xgb_params": XGB_PARAMS,
        "recursive_validation_enabled": RECURSIVE_VALIDATION_ENABLED,
        "recursive_validation_days": RECURSIVE_VALIDATION_DAYS,
        "cluster_gating_enabled": CLUSTER_GATING_ENABLED,
        "cluster_gate_margin": CLUSTER_GATE_MARGIN,
        "include_profile_features": INCLUDE_PROFILE_FEATURES,
        "include_seasonal_priors": INCLUDE_SEASONAL_PRIORS,
        "plot_sample_per_group": PLOT_SAMPLE_PER_GROUP,
        "plot_max_groups": PLOT_MAX_GROUPS,
        "random_state": RANDOM_STATE,
    }


def run_experiment_subprocess(exp_config: dict, run_settings: dict):
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath

    cmd = [
        sys.executable,
        "-m",
        "forecasting.experiment",
        "--repo-root",
        str(REPO_ROOT),
        "--config-json",
        json.dumps(exp_config),
        "--settings-json",
        json.dumps(run_settings),
    ]

    try:
        subprocess.run(cmd, check=True, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Experiment failed: {exp_config['experiment_name']}") from exc


def load_experiment_metric(exp_name: str, table_name: str) -> pd.DataFrame:
    metric_path = ensure_experiment_dirs(REPO_ROOT, exp_name)["metrics"] / f"{table_name}.csv"
    if not metric_path.exists():
        raise FileNotFoundError(f"Missing metric output for {exp_name}: {metric_path}")
    return pd.read_csv(metric_path)

In [5]:
cluster_case_paths = discover_cluster_cases(CLUSTER_CASE_DIR)
experiment_configs = build_experiment_configs(cluster_case_paths)
run_settings = build_run_settings()

experiment_plan = pd.DataFrame(experiment_configs)
display(experiment_plan)
print(f"Planned experiments: {len(experiment_configs)}")

,case_name,variant_name,experiment_name,cluster_path,static_features_path
0,case5,base,case5_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None


Planned experiments: 1


In [6]:
completed_experiments = []

for exp_config in experiment_configs:
    print(f"\n=== Running {exp_config['experiment_name']} ===")
    run_experiment_subprocess(exp_config, run_settings)
    completed_experiments.append(exp_config["experiment_name"])


=== Running case5_base ===


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:24:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Building training rows: 100%|██████████| 17547/17547 [00:33<00:00, 523.11it/s]


[0]	validation_0-mae:7.55061
[50]	validation_0-mae:3.83816
[100]	validation_0-mae:2.78202
[150]	validation_0-mae:2.58114
[200]	validation_0-mae:2.54017
[250]	validation_0-mae:2.52010
[300]	validation_0-mae:2.51067
[350]	validation_0-mae:2.49155
[400]	validation_0-mae:2.48298
[450]	validation_0-mae:2.47107
[500]	validation_0-mae:2.46325
[550]	validation_0-mae:2.45679
[600]	validation_0-mae:2.45117
[650]	validation_0-mae:2.44574
[700]	validation_0-mae:2.44110
[750]	validation_0-mae:2.43593
[800]	validation_0-mae:2.43187
[850]	validation_0-mae:2.42770
[899]	validation_0-mae:2.42426


Forecasting by cluster: 100%|██████████| 366/366 [03:29<00:00,  1.75it/s]


In [7]:
all_overall_summary = pd.concat(
    [load_experiment_metric(exp_name, "overall_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_route_summary = pd.concat(
    [load_experiment_metric(exp_name, "route_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_mae_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_mae_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_compare_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_compare_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_monthly_error_summary = pd.concat(
    [load_experiment_metric(exp_name, "monthly_error_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_recursive_validation_summary = pd.concat(
    [load_experiment_metric(exp_name, "recursive_validation_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_route_decisions = pd.concat(
    [load_experiment_metric(exp_name, "cluster_route_decisions") for exp_name in completed_experiments],
    ignore_index=True,
)

model_comparison = (
    all_overall_summary
    .pivot(index="experiment_name", columns="model", values="mean_mae")
    .reset_index()
)
model_comparison["delta_cluster_minus_global"] = (
    model_comparison[f"cluster_{MODEL_NAME}"] - model_comparison[f"global_{MODEL_NAME}"]
)
model_comparison = model_comparison.sort_values("delta_cluster_minus_global").reset_index(drop=True)

all_overall_summary.to_csv(OUT["metrics"] / "all_experiments_overall_summary.csv", index=False)
all_route_summary.to_csv(OUT["metrics"] / "all_experiments_route_summary.csv", index=False)
all_cluster_mae_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_mae_summary.csv", index=False)
all_cluster_compare_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_compare_summary.csv", index=False)
all_monthly_error_summary.to_csv(OUT["metrics"] / "all_experiments_monthly_error_summary.csv", index=False)
all_recursive_validation_summary.to_csv(OUT["metrics"] / "all_experiments_recursive_validation_summary.csv", index=False)
all_cluster_route_decisions.to_csv(OUT["metrics"] / "all_experiments_cluster_route_decisions.csv", index=False)
model_comparison.to_csv(OUT["metrics"] / "all_experiments_model_comparison.csv", index=False)

display(all_overall_summary.sort_values(["experiment_name", "model"]).reset_index(drop=True))
display(model_comparison)
display(all_recursive_validation_summary.sort_values("selected_score").reset_index(drop=True))
display(all_monthly_error_summary.sort_values(["experiment_name", "model", "month"]).reset_index(drop=True))
display(all_cluster_route_decisions.sort_values(["experiment_name", "xgb_profile", "mean_delta"]).reset_index(drop=True))
display(all_cluster_compare_summary.head(20))

,n_households,mean_mae,median_mae,std_mae,model,experiment_name
0,17547,3.368432,2.071450,4.330635,cluster_xgb,case5_base
1,17547,3.413474,2.099406,4.405548,global_xgb,case5_base


model,experiment_name,cluster_xgb,global_xgb,delta_cluster_minus_global
0,case5_base,3.368432,3.413474,-0.045041


,xgb_profile,validation_days,validation_start,validation_end,global_mae,cluster_mae,gated_cluster_mae,selected_score,n_features,n_trained_cluster_models,n_allowed_cluster_models,experiment_name
0,shallow,60,2023-11-02,2023-12-31,4.414310,4.474053,4.358093,4.358093,80,9,4,case5_base
1,regularized,60,2023-11-02,2023-12-31,4.480757,4.552854,4.463631,4.463631,80,9,4,case5_base
2,deeper,60,2023-11-02,2023-12-31,4.492372,4.528207,4.466793,4.466793,80,9,3,case5_base


,month,n_days,n_households,mae,bias_pred_minus_actual,actual_mean,pred_mean,model,experiment_name
0,1,31,17547,3.656181,-0.936452,12.544761,11.608307,cluster_xgb,case5_base
1,2,29,17547,3.814481,1.460272,10.227504,11.687779,cluster_xgb,case5_base
2,3,31,17547,3.644355,1.331153,9.285979,10.617134,cluster_xgb,case5_base
3,4,30,17547,3.927521,1.807181,8.176587,9.983767,cluster_xgb,case5_base
4,5,31,17547,3.217852,1.060301,7.232923,8.293223,cluster_xgb,case5_base
5,6,30,17547,2.715797,0.180237,7.177243,7.357480,cluster_xgb,case5_base
6,7,31,17547,2.625226,0.140281,7.083077,7.223359,cluster_xgb,case5_base
7,8,31,17547,2.713462,0.267725,7.109647,7.377371,cluster_xgb,case5_base
8,9,30,17547,3.022951,-0.561930,7.796201,7.234272,cluster_xgb,case5_base
9,10,31,17547,3.168579,-0.672379,8.663812,7.991432,cluster_xgb,case5_base


,ForecastGroup,RefinedCluster,n_households,mean_mae_global,mean_mae_cluster,mean_delta,median_delta,use_cluster_model,route_decision,xgb_profile,experiment_name
0,2,2,8465,2.498777,2.454430,-0.044347,-0.006048,True,cluster_model,deeper,case5_base
1,0,0,2827,2.956662,2.931610,-0.025052,0.002803,True,cluster_model,deeper,case5_base
2,8,8,198,1.551993,1.538719,-0.013273,-0.022716,True,cluster_model,deeper,case5_base
3,1,1,5351,8.593014,8.683366,0.090352,0.006501,False,global_fallback,deeper,case5_base
4,4,4,280,2.551030,2.720241,0.169211,0.000811,False,global_fallback,deeper,case5_base
5,6,6,86,0.784928,1.072502,0.287574,0.000499,False,global_fallback,deeper,case5_base
6,7,7,46,6.987470,7.552466,0.564996,0.022335,False,global_fallback,deeper,case5_base
7,9,9,51,10.887604,12.603737,1.716133,1.466069,False,global_fallback,deeper,case5_base
8,5,5,84,16.021678,20.885286,4.863608,2.764302,False,global_fallback,deeper,case5_base
9,7,7,46,7.175837,7.012152,-0.163685,-0.061899,True,cluster_model,regularized,case5_base


,ForecastGroup,RefinedCluster,n_households,mean_mae_global,mean_mae_cluster,mean_delta,median_delta,experiment_name
0,6,6,86,1.912520,1.690935,-0.221585,-0.000975,case5_base
1,2,2,8465,2.313306,2.220679,-0.092627,-0.019099,case5_base
2,4,4,280,2.096031,2.108140,0.012109,-0.000042,case5_base
3,8,8,198,1.301510,1.349027,0.047517,0.003029,case5_base
